In [1]:
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [2]:
data_path = "dataset/data_log.csv"
df = pd.read_csv(data_path)
df.head()

,id,maNguoiDung,ip,time,location,device,label
0,1,ND005,113.160.12.35,2026-05-01 08:10:12,"16.054200,108.202100",android 13,1
1,2,ND005,113.160.12.35,2026-05-01 19:18:44,"16.054240,108.202130",android 13,1
2,3,ND005,113.160.12.35,2026-05-02 08:05:33,"16.054210,108.202090",android 14,1
3,4,ND005,113.160.12.35,2026-05-02 19:21:16,"16.054260,108.202160",android 14,1
4,5,ND005,113.160.12.35,2026-05-03 08:12:09,"16.054230,108.202110",android 13,1


In [3]:
def parse_location(value):
    if pd.isna(value):
        return None, None
    if isinstance(value, str) and "," in value:
        parts = value.split(",")
        if len(parts) >= 2:
            try:
                return float(parts[0].strip()), float(parts[1].strip())
            except ValueError:
                return None, None
    return None, None

def build_features(frame):
    frame = frame.copy()
    frame["time"] = pd.to_datetime(frame["time"], errors="coerce")
    frame["hour"] = frame["time"].dt.hour.fillna(0).astype(int)
    frame["weekday"] = frame["time"].dt.weekday.fillna(0).astype(int)

    lat_lon = frame["location"].apply(parse_location)
    frame["lat"] = pd.to_numeric(lat_lon.str[0], errors="coerce").fillna(0.0)
    frame["lon"] = pd.to_numeric(lat_lon.str[1], errors="coerce").fillna(0.0)

    frame["ip"] = frame["ip"].fillna("")
    frame["device"] = frame["device"].fillna("")
    frame["maNguoiDung"] = frame["maNguoiDung"].fillna("")

    features = frame[["ip", "device", "maNguoiDung", "hour", "weekday", "lat", "lon"]]
    features = pd.get_dummies(features, columns=["ip", "device", "maNguoiDung"], drop_first=False)
    return features

X = build_features(df)
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
 )

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         4
           1       1.00      1.00      1.00        16

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [4]:
model_bundle = {
    "model": model,
    "columns": list(X.columns),
}
joblib.dump(model_bundle, "model.joblib")
print("Saved model.joblib")

Saved model.joblib
